# Featuresmith Tutorial: 05 — Comparing Dataset Versions: Dataset Diff & Diff-Aware Review (v0.4.0)

Compare dataset snapshot versions with the lower-level `fs.diff()` primitive AND the integrated diff-aware review `fs.review(..., previous=...)` (v0.3.0+, via the `DiffReviewer`), to prevent silent schema drift, missingness spikes, and quality regressions.

---


## 1. Why Dataset Diffing Matters
In production ML pipelines, datasets evolve continuously. New snapshots arrive daily or weekly. Silent changes — such as dropped columns, renamed features, type shifts, or missing value spikes — can break model inference or corrupt retrained models.

Featuresmith offers two levels of comparison:
- **`fs.diff(old, new)`** — the standalone, lower-level comparison primitive. It compares two snapshots deterministically and returns a `DatasetDiffResult` with an overall health verdict:
  - **`unchanged`**: No material structural or quality changes.
  - **`improved`**: Quality metrics improved (e.g., missingness decreased, leakage eliminated).
  - **`regressed`**: Quality degraded (e.g., columns dropped, missingness spiked, schema broke).
- **`fs.review(new, previous=old)`** (v0.3.0) — a full dataset review with the dataset diff integrated through the **`DiffReviewer`**. The diff becomes one more review section (`review.diff`) inside the normal `ReviewResult`, so you get the review and the diff in one call. `fs.diff()` stays as the lower-level primitive; `DiffReviewer` reuses it internally.

### Prerequisite: Prepare the Sales Dataset
This notebook loads `examples/data/processed/sales.csv`, which `examples/prepare_datasets.py` generates deterministically (no network). From the repository root, run:

```bash
python examples/prepare_datasets.py
```


### Step 1: Simulate Dataset Evolution (Snapshot v1 vs Snapshot v2)

In [1]:
import os

import pandas as pd

import featuresmith as fs

data_path = os.path.join("..", "data", "processed", "sales.csv")
v1 = pd.read_csv(data_path)

v2 = v1.copy()
v2.drop(columns=["store_version"], inplace=True)
v2["promo_code"] = "SUMMER2026"
v2.loc[:50, "discount"] = None

print(f"Snapshot v1 Shape: {v1.shape}")
print(f"Snapshot v2 Shape: {v2.shape}")

Snapshot v1 Shape: (1000, 10)
Snapshot v2 Shape: (1000, 10)


### Step 2: Execute Dataset Diff Engine (`fs.diff`)

In [2]:
diff_result = fs.diff(v1, v2)

print(f"Health Verdict : {diff_result.summary.overall_health.upper()}")
print(f"Recommendation : {diff_result.summary.recommendation}")
print(f"Added Columns  : {diff_result.schema.added_columns}")
print(f"Removed Columns: {diff_result.schema.removed_columns}")
print(f"Missingness Shift Count: {diff_result.summary.missing_values_increased}")

Health Verdict : REGRESSED
Recommendation : Dataset regressed: 1 column(s) removed; missingness increased in 1 column(s). Review the changes before retraining.
Added Columns  : ('promo_code',)
Removed Columns: ('store_version',)
Missingness Shift Count: 1


### Step 3: Extract Diff Findings & Render Diff Text Report
Use `fs.diff_findings()` to extract `RuleFinding` objects from a diff result, and `fs.render_diff()` for terminal output.

In [3]:
findings = fs.diff_findings(diff_result)
print(f"Derived Diff Findings Count: {len(findings)}")
for f in findings[:3]:
    print(f"  - [{f.severity.upper()}] {f.title}")

diff_report = fs.render_diff(diff_result, target="console")
print("\n=== Formatted Diff Text Report Preview ===\n")
print(diff_report[:500] + "\n...")

Derived Diff Findings Count: 3
  - [INFO] Columns added to the dataset
  - [WARNING] Columns removed from the dataset
  - [WARNING] Missing values increased in column 'discount'

=== Formatted Diff Text Report Preview ===

Featuresmith Dataset Diff
Rows: 1,000 -> 1,000 (+0 / -0) | Columns: 10 -> 10
Engine: v0.2.0

Rows 0 removed, 0 added; columns 1 removed, 1 added; overall health: regressed.

Dataset Comparison Summary
  Rows Added: 0
  Rows Removed: 0
  Columns Added: 1
  Columns Removed: 1
  Columns Renamed: 0
  Schema Changed: Yes
  Type Changes: 0
  Missing Values Increased: 1 column(s)
  Missing Values Improved: 0 column(s)
  Duplicate Rows Increased: No
  Duplicate Rows Improved: No
  Newly Constant Columns
...


### Step 4: Diff-Aware Review with `fs.review(..., previous=...)` (v0.3.0)

Here `v1` is the **previous** snapshot (the baseline already in production) and `v2` is the **current** snapshot (the candidate about to replace it).
Passing `previous=v1` to `fs.review()` activates the **`DiffReviewer`**: the same diff that `fs.diff()` computed now appears as a `review.diff` section inside the normal review, so one call returns both the full dataset review **and** the snapshot comparison.

What the `DiffReviewer` reports:
- Columns **added** to the dataset (new features, possibly unknown to a trained model).
- Columns **removed** from the dataset (features a model was trained on that no longer exist).
- Quality regressions between snapshots (e.g., missing values increasing in a column).

The diff verdict is **informational**: it does not change the 8 ML Readiness Score dimensions.

In [4]:
review_res = fs.review(v2, previous=v1)

print(
    f"Review Sections          : {len(review_res.sections)} (8 built-in + review.diff)"
)
print(f"Overall Summary          : {review_res.overall_summary}")

diff_section = next(s for s in review_res.sections if s.id == "review.diff")
print(f"Diff Section             : {diff_section.title} [{diff_section.severity}]")
print("DiffReviewer Findings   :")
for finding in diff_section.findings:
    print(f"  - [{finding.severity.upper():<7}] {finding.title}")

print("The DatasetDiffResult is also attached to the review result:")
print(
    f"  ReviewResult.diff Health       : {review_res.diff.summary.overall_health.upper()}"
)
print(
    f"  ReviewResult.diff Added/Removed: {review_res.diff.schema.added_columns} / {review_res.diff.schema.removed_columns}"
)
if review_res.score:
    print(
        f"  ML Readiness Score             : {review_res.score.overall:.1f}/100 (8 dimensions; diff is informational)"
    )

Review Sections          : 10 (8 built-in + review.diff)
Overall Summary          : 5 of 10 sections passed with 8 finding(s) identified across the review.
Diff Section             : Dataset Diff [Severity.WARNING]
DiffReviewer Findings   :
  - [INFO   ] Columns added to the dataset
  - [WARNING] Columns removed from the dataset
  - [WARNING] Missing values increased in column 'discount'
The DatasetDiffResult is also attached to the review result:
  ReviewResult.diff Health       : REGRESSED
  ReviewResult.diff Added/Removed: ('promo_code',) / ('store_version',)
  ML Readiness Score             : 90.0/100 (8 dimensions; diff is informational)


### Step 5: Same Workflow from the CLI
The previous-snapshot review is also available from the terminal — write the two snapshots to CSV, then run:

```bash
featuresmith review sales_v2.csv --previous sales_v1.csv
```

The CLI produces the identical 9-section `ReviewResult`. Exit codes are unchanged: `0` clean, `1` findings at or above the `--fail-on` threshold, `2` invalid input, `3` file load/parse failure. The cell below runs the real CLI against the snapshots we built in Step 1.

In [5]:
import json
import subprocess
import sys
import tempfile

tmp_dir = tempfile.mkdtemp()
v1_path = os.path.join(tmp_dir, "sales_v1.csv")
v2_path = os.path.join(tmp_dir, "sales_v2.csv")
v1.to_csv(v1_path, index=False)
v2.to_csv(v2_path, index=False)

proc = subprocess.run(
    [
        sys.executable,
        "-m",
        "featuresmith_cli.main",
        "review",
        v2_path,
        "--previous",
        v1_path,
        "--format",
        "json",
    ],
    capture_output=True,
    text=True,
    timeout=120,
)

cli_result = json.loads(proc.stdout)
diff_sections = [s for s in cli_result["sections"] if s["id"] == "review.diff"]
print(f"CLI Exit Code      : {proc.returncode}")
print(f"Review Sections    : {len(cli_result['sections'])}")
print(f"Diff Section       : {diff_sections[0]['severity']}")
print(f"Diff Findings      : {[f['rule_id'] for f in diff_sections[0]['findings']]}")
print(f"Diff Health        : {cli_result['diff']['summary']['overall_health']}")

CLI Exit Code      : 1
Review Sections    : 10
Diff Section       : warning
Diff Findings      : ['diff.schema.added_columns', 'diff.schema.removed_columns', 'diff.quality.missing_increased']
Diff Health        : regressed


### Key Takeaways & Connection to Next Tutorial
- `fs.diff()` is the lower-level primitive: a direct, standalone snapshot comparison with an immediate pass/fail verdict.
- `fs.review(new, previous=old)` (v0.3.0) integrates the diff into a full review via the `DiffReviewer`, so one call returns both the review and the diff.
- `fs.diff_findings()` converts diff deltas into standard `RuleFinding` objects for CI exit code gating.

**Next Tutorial**: In `06_end_to_end_workflow.ipynb`, we build a production pre-training pipeline gate that integrates review, scoring, and error handling.